# COLM Norm Reasoning Traces — Exploratory Analysis

Explores the reasoning output from the Qwen2.5-72B-AWQ extraction of Raz norms
from 10 fiction novels. Each row is one identified norm (exploded from chunk-level output).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv(os.path.join(os.path.dirname(os.getcwd()), '.env'))

REASONING_PATH = os.environ.get('REASONING_PATH',
    '/share/pierson/matt/UAIR/outputs/2026-03-20_historical_norms/23-12-53/COLM_norms_fiction/outputs/reasoning/reasoning.parquet')

df = pd.read_parquet(REASONING_PATH)
print(f'Loaded {len(df):,} rows from {REASONING_PATH}')
print(f'Columns: {list(df.columns)}')
df.head(2)

## 1. Corpus overview

In [ ]:
n_chunks = df[['gutenberg_id', 'chunk_id']].drop_duplicates().shape[0]
n_norms = df['has_norms'].sum()
n_empty = (~df['has_norms']).sum()
n_errors = df['reasoning_error'].notna().sum()

print(f'Total rows (one per norm):  {len(df):,}')
print(f'Unique chunks processed:    {n_chunks:,}')
print(f'Rows with norms:            {n_norms:,} ({n_norms/len(df)*100:.1f}%)')
print(f'Rows without norms:         {n_empty:,} ({n_empty/len(df)*100:.1f}%)')
print(f'Parse errors:               {n_errors:,}')
print(f'Avg norms per chunk:        {n_norms / n_chunks:.1f}')

## 2. Norms per novel

In [ ]:
norms_by_book = df[df['has_norms']].groupby('book_title').agg(
    total_norms=('norm_index', 'count'),
    chunks_with_norms=('chunk_id', 'nunique'),
    info_flow_norms=('governs_information_flow', lambda x: (x == True).sum() + (x == 'True').sum()),
).sort_values('total_norms', ascending=True)

# Also get total chunks per book
total_chunks = df.groupby('book_title')[['gutenberg_id', 'chunk_id']].drop_duplicates().groupby('book_title').size()
norms_by_book['total_chunks'] = total_chunks
norms_by_book['norms_per_chunk'] = (norms_by_book['total_norms'] / norms_by_book['total_chunks']).round(1)
norms_by_book['pct_chunks_with_norms'] = (norms_by_book['chunks_with_norms'] / norms_by_book['total_chunks'] * 100).round(1)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

norms_by_book['total_norms'].plot.barh(ax=axes[0], color='steelblue')
axes[0].set_xlabel('Total norms extracted')
axes[0].set_title('Norms per Novel')

norms_by_book['norms_per_chunk'].plot.barh(ax=axes[1], color='darkorange')
axes[1].set_xlabel('Avg norms per chunk')
axes[1].set_title('Norm Density')

norms_by_book['pct_chunks_with_norms'].plot.barh(ax=axes[2], color='seagreen')
axes[2].set_xlabel('% chunks with norms')
axes[2].set_title('Coverage')
axes[2].axvline(100, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

norms_by_book.sort_values('total_norms', ascending=False)

## 3. Normative force distribution

In [ ]:
force_order = ['obligatory', 'recommended', 'prohibited', 'permitted', 'discouraged']
force_colors = {'obligatory': '#d62728', 'recommended': '#2ca02c', 'prohibited': '#1f77b4',
                'permitted': '#ff7f0e', 'discouraged': '#9467bd'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
force_counts = df['preliminary_normative_force'].value_counts().reindex(force_order)
force_counts.plot.bar(ax=axes[0], color=[force_colors.get(f, 'gray') for f in force_order])
axes[0].set_title('Normative Force — All Novels')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Per novel (stacked)
force_by_book = pd.crosstab(df['book_title'], df['preliminary_normative_force'])
force_by_book = force_by_book.reindex(columns=force_order, fill_value=0)
force_by_book_pct = force_by_book.div(force_by_book.sum(axis=1), axis=0) * 100
force_by_book_pct.sort_index().plot.barh(
    stacked=True, ax=axes[1],
    color=[force_colors.get(f, 'gray') for f in force_order]
)
axes[1].set_xlabel('% of norms')
axes[1].set_title('Normative Force Distribution by Novel')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()

## 4. Information flow norms vs. non-informational norms

In [ ]:
# Normalize the governs_information_flow column
df['info_flow'] = df['governs_information_flow'].apply(
    lambda x: True if x is True or x == 'True' or x == 'true' else False
)

flow_by_book = df[df['has_norms']].groupby('book_title')['info_flow'].agg(
    info_flow_count='sum',
    total='count'
)
flow_by_book['non_info_count'] = flow_by_book['total'] - flow_by_book['info_flow_count']
flow_by_book['info_flow_pct'] = (flow_by_book['info_flow_count'] / flow_by_book['total'] * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))
flow_by_book[['non_info_count', 'info_flow_count']].sort_index().plot.barh(
    stacked=True, ax=ax, color=['#7f7f7f', '#e377c2']
)
ax.set_xlabel('Number of norms')
ax.set_title('Information Flow Norms vs. Non-Informational Norms')
ax.legend(['Non-informational', 'Information flow'], loc='lower right')
plt.tight_layout()
plt.show()

total_info = df['info_flow'].sum()
print(f'Information flow norms: {total_info:,} / {n_norms:,} ({total_info/n_norms*100:.1f}%)')
print()
flow_by_book.sort_values('info_flow_pct', ascending=False)

## 5. Norm count distribution per chunk

In [ ]:
norms_per_chunk = df.groupby(['gutenberg_id', 'chunk_id']).agg(
    n_norms=('has_norms', 'sum'),
    book_title=('book_title', 'first')
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

norms_per_chunk['n_norms'].hist(bins=range(0, 15), ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Norms per chunk')
axes[0].set_ylabel('Number of chunks')
axes[0].set_title('Distribution of Norms per Chunk')

norms_per_chunk.boxplot(column='n_norms', by='book_title', ax=axes[1], vert=False)
axes[1].set_xlabel('Norms per chunk')
axes[1].set_title('Norms per Chunk by Novel')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f'Chunks with 0 norms: {(norms_per_chunk["n_norms"] == 0).sum()} / {len(norms_per_chunk)}')
print(f'Max norms in a single chunk: {norms_per_chunk["n_norms"].max()}')

## 6. Sample reasoning traces

Qualitative inspection of a few high-quality reasoning traces.

In [ ]:
# Pick one norm from each of 5 diverse novels
sample_books = ['Pride and Prejudice', 'Nineteen Eighty-Four', 'Les Misérables',
                'Anna Karenina', 'Alice\'s Adventures in Wonderland']

for book in sample_books:
    subset = df[(df['book_title'] == book) & (df['has_norms'])]
    if subset.empty:
        continue
    row = subset.sample(1, random_state=42).iloc[0]
    print(f'{"═" * 70}')
    print(f'{book} — chunk {row["chunk_id"]}, norm {int(row.get("norm_index", 0))}')
    print(f'Force: {row["preliminary_normative_force"]} | Info flow: {row["governs_information_flow"]}')
    print(f'{"─" * 70}')
    print(f'Snippet: {str(row.get("norm_snippet", ""))[:300]}')
    print(f'{"─" * 70}')
    trace = str(row.get('reasoning_trace', ''))
    print(f'Reasoning ({len(trace)} chars):')
    print(trace[:500])
    if len(trace) > 500:
        print('...')
    print()

## 7. Empty chunks — what text has no norms?

In [ ]:
empty_chunks = df[~df['has_norms']][['book_title', 'chunk_id', 'article_text']].drop_duplicates()

print(f'Chunks with no norms: {len(empty_chunks)}')
print()
print('Distribution by book:')
print(empty_chunks['book_title'].value_counts().to_string())
print()
print('Sample empty chunks (first 200 chars):')
for _, row in empty_chunks.sample(min(5, len(empty_chunks)), random_state=42).iterrows():
    print(f'  [{row["book_title"]}, chunk {row["chunk_id"]}]: {row["article_text"][:200]}...')
    print()

## 8. Parse errors

In [ ]:
errors = df[df['reasoning_error'].notna()]
print(f'Parse errors: {len(errors)}')
if len(errors) > 0:
    for _, row in errors.iterrows():
        print(f'  [{row["book_title"]}, chunk {row["chunk_id"]}]: {row["reasoning_error"]}')
        print(f'  Raw output (first 300 chars): {str(row["generated_text"])[:300]}')
        print()

## 9. Reasoning trace length distribution

In [ ]:
df['trace_len'] = df['reasoning_trace'].fillna('').str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df[df['has_norms']]['trace_len'].hist(bins=50, ax=axes[0], edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Reasoning trace length (chars)')
axes[0].set_ylabel('Count')
axes[0].set_title('Reasoning Trace Length Distribution')

df[df['has_norms']].boxplot(column='trace_len', by='book_title', ax=axes[1], vert=False)
axes[1].set_xlabel('Trace length (chars)')
axes[1].set_title('Trace Length by Novel')
plt.suptitle('')

plt.tight_layout()
plt.show()

print(f'Trace length stats: mean={df["trace_len"].mean():.0f}, '
      f'median={df["trace_len"].median():.0f}, '
      f'max={df["trace_len"].max()}')